# Milestone 3: pure-greedy re-evaluation, class-balanced v2, draft model, KV cache, tiled attention, speculative decoding
Kaggle: **GPU T4 x2, Internet ON**, Run All. About 60-75 min.

**Before running:** put your HuggingFace adapter id in the first code cell.

In [1]:
   HF_ADAPTER_V1 = 'usmanbari2005/trialmatch-qwen25-1p5b-lora'

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
assert 'YOUR_HF_USERNAME' not in HF_ADAPTER_V1, 'set HF_ADAPTER_V1 first'
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [2]:
%cd /kaggle/working
!rm -rf trial-eligibility-transformer
!git clone https://github.com/UsmanBari/trial-eligibility-transformer.git
%cd /kaggle/working/trial-eligibility-transformer
!pip install -q -r requirements.txt && pip install -q -e .
!python -m pytest -q tests   # includes KV / speculative / tiled-attention tests on tiny models
!python scripts/prepare_data.py | tail -3

/kaggle/working
Cloning into 'trial-eligibility-transformer'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 138 (delta 53), reused 128 (delta 43), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 422.19 KiB | 3.84 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/kaggle/working/trial-eligibility-transformer
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
litellm 1.93.2 requires

## 1. Re-evaluate with pure greedy decoding
Qwen's default generation config adds a repetition penalty even when sampling is off. Our KV-cache and speculative decoders are pure greedy, so all evaluations are redone with `repetition_penalty=1.0` (a decoding-consistency fix, not tuning).

In [3]:
!python scripts/evaluate.py --split val --tag baseline_val_greedy --model Qwen/Qwen2.5-1.5B-Instruct
!python scripts/evaluate.py --split val --tag ft_v1_val_greedy --adapter {HF_ADAPTER_V1}
!python scripts/evaluate.py --split test --final-test --tag baseline_test_greedy --model Qwen/Qwen2.5-1.5B-Instruct
!python scripts/evaluate.py --split test --final-test --tag ft_v1_test_greedy --adapter {HF_ADAPTER_V1}

tokenizer_config.json: 7.30kB [00:00, 34.5MB/s]
vocab.json: 2.78MB [00:00, 42.9MB/s]
merges.txt: 1.67MB [00:00, 108MB/s]
tokenizer.json: 7.03MB [00:00, 185MB/s]
config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 4.46MB/s]
I0000 00:00:1793882796.771709     240 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
model.safetensors: 100%|████████████████████| 3.09G/3.09G [00:08<00:00, 343MB/s]
baseline_val_greedy:val: 100%|██████████████████| 26/26 [00:45<00:00,  1.77s/it]

n=206  acc=0.189  macroF1=0.143  invalid=0  | evidence microF1=0.190  macroF1(all)=0.393  macroF1(nonempty gold)=0.254
inclusion acc=0.500  exclusion acc=0.043
truncation: {'n_truncated': 0, 'n_gold_evidence_lost': 0, 'max_prompt_tokens': 479, 'mean_prompt_tokens': 372.2330097087379}  parse_failures=0
saved -> /kaggle/working/trial-eligibility-transformer/results/baseline_val_greedy
I0000 00:00:1793882880.177643     300 cudart_stub.cc:31] Could not find cuda drivers 

## 2. v2: class-balanced fine-tuning (train split oversampled, 2 epochs), selected on validation

In [4]:
!python scripts/train_lora.py --balance sqrt --epochs 2 --output-dir outputs/lora_qwen25_1p5b_v2

I0000 00:00:1793883048.558330     393 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
class balancing (type, label): count -> copies: {'exclusion/excluded': [8, 4], 'exclusion/not applicable': [23, 4], 'exclusion/not enough information': [65, 2], 'exclusion/not excluded': [291, 1], 'inclusion/included': [88, 2], 'inclusion/not applicable': [5, 4], 'inclusion/not enough information': [109, 2], 'inclusion/not included': [13, 4]}
train examples: 1011 (skipped 0 over max_seq_len)  val: 206  max len: 667  mean len: 373  mean target tokens: 16.6
trainable params: 18,464,768 / 1,562,179,072 (1.18%)
{'loss': 0.3479, 'grad_norm': 1.3332175016403198, 'learning_rate': 0.00014285714285714287, 'epoch': 0.08}
{'loss': 0.2205, 'grad_norm': 0.5617367625236511, 'learning_rate': 0.00019968653337272261, 'epoch': 0.16}
{'loss': 0.1596, 'grad_norm': 0.42050549387931824, 'learning_rate': 0.00019777801622550408, 'epoch': 0.24}
{'loss': 0.16, 'grad_norm': 0.61958789825439

In [5]:
import glob, subprocess, json
ck = sorted(glob.glob('outputs/lora_qwen25_1p5b_v2/checkpoint-*'), key=lambda p: int(p.split('-')[-1]))
assert ck, 'v2 training failed'
tags = []
for i, c in enumerate(ck, 1):
    tags.append(f'ft_v2_ep{i}_val')
    subprocess.run(['python', 'scripts/evaluate.py', '--adapter', c, '--split', 'val', '--tag', tags[-1]], check=True)
subprocess.run(['python', 'scripts/select_checkpoint.py', '--run', 'outputs/lora_qwen25_1p5b_v2', '--tags', *tags,
                '--selection-file', 'checkpoint_selection_v2.json'], check=True)

I0000 00:00:1793884035.985861     421 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
ft_v2_ep1_val:val: 100%|██████████| 26/26 [00:34<00:00,  1.32s/it]



n=206  acc=0.743  macroF1=0.547  invalid=0  | evidence microF1=0.488  macroF1(all)=0.651  macroF1(nonempty gold)=0.656
inclusion acc=0.697  exclusion acc=0.764
truncation: {'n_truncated': 0, 'n_gold_evidence_lost': 0, 'max_prompt_tokens': 479, 'mean_prompt_tokens': 372.2330097087379}  parse_failures=0
saved -> /kaggle/working/trial-eligibility-transformer/results/ft_v2_ep1_val


I0000 00:00:1793884097.459815     446 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
ft_v2_ep2_val:val: 100%|██████████| 26/26 [00:32<00:00,  1.26s/it]



n=206  acc=0.762  macroF1=0.581  invalid=0  | evidence microF1=0.529  macroF1(all)=0.711  macroF1(nonempty gold)=0.616
inclusion acc=0.697  exclusion acc=0.793
truncation: {'n_truncated': 0, 'n_gold_evidence_lost': 0, 'max_prompt_tokens': 479, 'mean_prompt_tokens': 372.2330097087379}  parse_failures=0
saved -> /kaggle/working/trial-eligibility-transformer/results/ft_v2_ep2_val
epoch 1: acc=0.743 macroF1=0.547 evF1=0.488
epoch 2: acc=0.762 macroF1=0.581 evF1=0.529  <- selected
best adapter -> /kaggle/working/trial-eligibility-transformer/outputs/lora_qwen25_1p5b_v2_best


CompletedProcess(args=['python', 'scripts/select_checkpoint.py', '--run', 'outputs/lora_qwen25_1p5b_v2', '--tags', 'ft_v2_ep1_val', 'ft_v2_ep2_val', '--selection-file', 'checkpoint_selection_v2.json'], returncode=0)

In [6]:
# Final model choice: higher VALIDATION macro-F1 (rule fixed in advance)
v1 = json.load(open('results/ft_v1_val_greedy/metrics.json'))['overall']['classification']
sel = json.load(open('results/checkpoint_selection_v2.json'))['selected']
FINAL = 'v2' if sel['val_macro_f1'] > v1['macro_f1'] else 'v1'
FINAL_ADAPTER = 'outputs/lora_qwen25_1p5b_v2_best' if FINAL == 'v2' else HF_ADAPTER_V1
BALANCE = 'sqrt' if FINAL == 'v2' else 'none'
json.dump({'rule': 'higher validation macro-F1 under pure greedy decoding',
           'v1': {'adapter': HF_ADAPTER_V1, 'val_macro_f1': v1['macro_f1'], 'val_accuracy': v1['accuracy']},
           'v2': {'adapter': 'outputs/lora_qwen25_1p5b_v2_best', 'val_macro_f1': sel['val_macro_f1'], 'val_accuracy': sel['val_accuracy'], 'epoch': sel['epoch']},
           'final': FINAL}, open('results/final_model.json', 'w'), indent=2)
print(open('results/final_model.json').read())
if FINAL == 'v2':
    subprocess.run(['python', 'scripts/evaluate.py', '--split', 'test', '--final-test', '--tag', 'ft_v2_test_greedy', '--adapter', FINAL_ADAPTER], check=True)
FINAL_TEST_TAG = 'ft_v2_test_greedy' if FINAL == 'v2' else 'ft_v1_test_greedy'

{
  "rule": "higher validation macro-F1 under pure greedy decoding",
  "v1": {
    "adapter": "usmanbari2005/trialmatch-qwen25-1p5b-lora",
    "val_macro_f1": 0.4680981813015061,
    "val_accuracy": 0.7330097087378641
  },
  "v2": {
    "adapter": "outputs/lora_qwen25_1p5b_v2_best",
    "val_macro_f1": 0.5813278934157285,
    "val_accuracy": 0.7621359223300971,
    "epoch": 2
  },
  "final": "v2"
}


I0000 00:00:1793884157.781930     473 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
ft_v2_test_greedy:test: 100%|██████████| 26/26 [00:32<00:00,  1.24s/it]



n=206  acc=0.718  macroF1=0.632  invalid=0  | evidence microF1=0.575  macroF1(all)=0.804  macroF1(nonempty gold)=0.617
inclusion acc=0.787  exclusion acc=0.679
truncation: {'n_truncated': 0, 'n_gold_evidence_lost': 0, 'max_prompt_tokens': 544, 'mean_prompt_tokens': 341.00970873786406}  parse_failures=0
saved -> /kaggle/working/trial-eligibility-transformer/results/ft_v2_test_greedy


In [7]:
subprocess.run(['python', 'scripts/error_analysis.py', f'results/{FINAL_TEST_TAG}'], check=True)
tags, names = ['baseline_test_greedy', 'ft_v1_test_greedy'], ['Zero-shot', 'LoRA v1']
if FINAL == 'v2':
    tags.append('ft_v2_test_greedy'); names.append('LoRA v2 (balanced)')
subprocess.run(['python', 'scripts/make_figures.py', '--cm', 'baseline_test_greedy', *tags[1:],
                '--compare', *tags, '--compare-names', *names,
                '--train-log', 'outputs/lora_qwen25_1p5b_v2/train_log.json'], check=True)

{
 "inclusion_flip": 0,
 "exclusion_flip": 0,
 "overconfident": 30,
 "missed_information": 20,
 "applicability": 14,
 "invalid_output": 0,
 "evidence_missing": 9,
 "evidence_wrong": 4,
 "evidence_hallucinated": 0
}
10 cases written -> results/ft_v2_test_greedy/error_analysis.md
saved /kaggle/working/trial-eligibility-transformer/results/figures/training_curve_lora_qwen25_1p5b_v2.png
saved /kaggle/working/trial-eligibility-transformer/results/figures/cm_baseline_test_greedy_overall.png
saved /kaggle/working/trial-eligibility-transformer/results/figures/cm_baseline_test_greedy_inclusion.png
saved /kaggle/working/trial-eligibility-transformer/results/figures/cm_baseline_test_greedy_exclusion.png
saved /kaggle/working/trial-eligibility-transformer/results/figures/cm_ft_v1_test_greedy_overall.png
saved /kaggle/working/trial-eligibility-transformer/results/figures/cm_ft_v1_test_greedy_inclusion.png
saved /kaggle/working/trial-eligibility-transformer/results/figures/cm_ft_v1_test_greedy_exclu

CompletedProcess(args=['python', 'scripts/make_figures.py', '--cm', 'baseline_test_greedy', 'ft_v1_test_greedy', 'ft_v2_test_greedy', '--compare', 'baseline_test_greedy', 'ft_v1_test_greedy', 'ft_v2_test_greedy', '--compare-names', 'Zero-shot', 'LoRA v1', 'LoRA v2 (balanced)', '--train-log', 'outputs/lora_qwen25_1p5b_v2/train_log.json'], returncode=0)

## 3. Draft model for speculative decoding: Qwen2.5-0.5B-Instruct (same tokenizer), LoRA-tuned on the same train split (1 epoch)

In [8]:
!python scripts/train_lora.py --model Qwen/Qwen2.5-0.5B-Instruct --epochs 1 --balance {BALANCE} --output-dir outputs/lora_draft_0p5b
DRAFT_ADAPTER = sorted(glob.glob('outputs/lora_draft_0p5b/checkpoint-*'))[-1]
!python scripts/evaluate.py --split val --tag draft_0p5b_val --model Qwen/Qwen2.5-0.5B-Instruct --adapter {DRAFT_ADAPTER}
!python scripts/evaluate.py --split val --tag draft_0p5b_base_val --model Qwen/Qwen2.5-0.5B-Instruct

I0000 00:00:1793884220.760052     513 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
tokenizer_config.json: 7.30kB [00:00, 12.9MB/s]
vocab.json: 2.78MB [00:00, 44.3MB/s]
merges.txt: 1.67MB [00:00, 133MB/s]
tokenizer.json: 7.03MB [00:00, 186MB/s]
class balancing (type, label): count -> copies: {'exclusion/excluded': [8, 4], 'exclusion/not applicable': [23, 4], 'exclusion/not enough information': [65, 2], 'exclusion/not excluded': [291, 1], 'inclusion/included': [88, 2], 'inclusion/not applicable': [5, 4], 'inclusion/not enough information': [109, 2], 'inclusion/not included': [13, 4]}
train examples: 1011 (skipped 0 over max_seq_len)  val: 206  max len: 667  mean len: 373  mean target tokens: 16.6
config.json: 100%|█████████████████████████████| 659/659 [00:00<00:00, 2.85MB/s]
model.safetensors: 100%|██████████████████████| 988M/988M [00:02<00:00, 429MB/s]
generation_config.json: 100%|███████████████████| 242/242 [00:00<00:00, 988kB/s]
trainable pa

## 4. Task 4: attention (reference vs tiled online-softmax vs fused SDPA; plus the whole model running on tiled attention)

In [9]:
!python scripts/bench_attention.py --adapter {FINAL_ADAPTER}

[correctness] 72/72 configurations within tolerance; worst tiled-vs-ref fp32 = 3.81e-06, fp16 = 9.77e-04
[perf] L=  256  reference=0.60ms/9MB  tiled_128=2.24ms/10MB  torch_sdpa_fused=0.09ms/1MB
[perf] L=  512  reference=1.72ms/36MB  tiled_128=6.10ms/14MB  torch_sdpa_fused=0.18ms/2MB
[perf] L= 1024  reference=5.52ms/144MB  tiled_128=18.38ms/24MB  torch_sdpa_fused=0.35ms/3MB
[perf] L= 2048  reference=17.97ms/576MB  tiled_128=59.40ms/45MB  torch_sdpa_fused=0.83ms/6MB
[perf] L= 4096  reference=70.70ms/2304MB  tiled_128=219.96ms/87MB  torch_sdpa_fused=4.05ms/12MB
[perf] L= 8192  reference=320.68ms/9217MB  tiled_128=901.54ms/171MB  torch_sdpa_fused=17.30ms/24MB
I0000 00:00:1793884771.917007     617 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[model torch.float32] greedy tokens equal with tiled attention in every layer: 5/5; max logit diff 2.82e-03
[model torch.float16] greedy tokens equal with tiled attention in every layer: 5/5; max logit diff 1.80e

## 5. Task 3: KV-cached vs no-cache greedy decoding (fp16 benchmark grid + fp32 exactness)

In [10]:
!python scripts/bench_kv.py --adapter {FINAL_ADAPTER} --dtype float16
!python scripts/bench_kv.py --adapter {FINAL_ADAPTER} --dtype float32 --no-grid

I0000 00:00:1793884846.654947     642 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[task] no-cache == KV: 20/20   KV == HF generate: 20/20
[grid] in=  128 out=  32  no-cache     1295 ms  KV    1018 ms  x1.27  equal=True
[grid] in=  128 out= 128  no-cache     5742 ms  KV    4159 ms  x1.38  equal=True
[grid] in=  128 out= 256  no-cache    15542 ms  KV    8475 ms  x1.83  equal=False (first diff @ 150)
[grid] in=  512 out=  32  no-cache     3862 ms  KV    1098 ms  x3.52  equal=True
[grid] in=  512 out= 128  no-cache    16390 ms  KV    4237 ms  x3.87  equal=True
[grid] in=  512 out= 256  no-cache    34641 ms  KV    8309 ms  x4.17  equal=True
[grid] in= 1024 out=  32  no-cache     6774 ms  KV    1175 ms  x5.76  equal=True
[grid] in= 1024 out= 128  no-cache    28157 ms  KV    4267 ms  x6.60  equal=True
[grid] in= 1024 out= 256  no-cache    58977 ms  KV    8551 ms  x6.90  equal=True
saved /kaggle/working/trial-eligibility-transformer/results/bench/kv_fl

## 6. Task 5: speculative decoding (base vs tuned 0.5B draft, k = 2/4/6/8) + fp32 exactness

In [11]:
!python scripts/bench_spec.py --adapter {FINAL_ADAPTER} --drafts base=Qwen/Qwen2.5-0.5B-Instruct tuned=Qwen/Qwen2.5-0.5B-Instruct@{DRAFT_ADAPTER} --ks 2 4 6 8 --dtype float16
!python scripts/bench_spec.py --adapter {FINAL_ADAPTER} --drafts tuned=Qwen/Qwen2.5-0.5B-Instruct@{DRAFT_ADAPTER} --ks 4 --dtype float32 --n-task 10 --n-summary 3

I0000 00:00:1793885767.648640     692 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[baseline] task       312 tokens    28.2 tok/s  latency   553.6 ms/prompt
[baseline] summary    663 tokens    29.4 tok/s  latency  3760.4 ms/prompt
[base  k=2] task     equal 20/20  accept 0.60    20.9 tok/s  speedup x0.74  tok/target-pass 1.93
[base  k=2] summary  equal 6/6  accept 0.77    25.8 tok/s  speedup x0.88  tok/target-pass 2.49
[base  k=4] task     equal 20/20  accept 0.55    19.0 tok/s  speedup x0.67  tok/target-pass 2.64
[base  k=4] summary  equal 6/6  accept 0.64    22.8 tok/s  speedup x0.78  tok/target-pass 3.44
[base  k=6] task     equal 20/20  accept 0.48    16.9 tok/s  speedup x0.60  tok/target-pass 3.09
[base  k=6] summary  equal 6/6  accept 0.57    20.8 tok/s  speedup x0.71  tok/target-pass 4.25
[base  k=8] task     equal 20/20  accept 0.37    13.8 tok/s  speedup x0.49  tok/target-pass 3.15
[base  k=8] summary  equal 6/6  accept 0.49    18.3 tok

In [12]:
!python scripts/make_bench_figures.py
for t in ['baseline_test_greedy', 'ft_v1_test_greedy'] + (['ft_v2_test_greedy'] if FINAL == 'v2' else []) + ['ft_v1_val_greedy', 'draft_0p5b_val', 'draft_0p5b_base_val']:
    o = json.load(open(f'results/{t}/metrics.json'))['overall']
    print(f"{t:24s} acc={o['classification']['accuracy']:.3f} mF1={o['classification']['macro_f1']:.3f} evF1={o['evidence']['micro']['f1']:.3f}")

saved /kaggle/working/trial-eligibility-transformer/results/figures/kv_cache_benchmark.png
saved /kaggle/working/trial-eligibility-transformer/results/figures/attention_benchmark.png
saved /kaggle/working/trial-eligibility-transformer/results/figures/speculative_benchmark.png
baseline_test_greedy     acc=0.170 mF1=0.156 evF1=0.206
ft_v1_test_greedy        acc=0.728 mF1=0.465 evF1=0.526
ft_v2_test_greedy        acc=0.718 mF1=0.632 evF1=0.575
ft_v1_val_greedy         acc=0.733 mF1=0.468 evF1=0.524
draft_0p5b_val           acc=0.743 mF1=0.447 evF1=0.430
draft_0p5b_base_val      acc=0.228 mF1=0.151 evF1=0.000


## 7. Download `m3_results.zip` (results + figures + benchmark logs) and `m3_adapters.zip` (v2 + draft adapters)

In [13]:
!for d in outputs/*/; do [ -f $d/train_log.json ] && cp $d/train_log.json results/train_log_$(basename $d).json && cp $d/train_config.json results/train_config_$(basename $d).json; done
!zip -qr /kaggle/working/m3_results.zip results data/splits
!cd outputs && zip -qr /kaggle/working/m3_adapters.zip lora_qwen25_1p5b_v2_best $(basename $(dirname {DRAFT_ADAPTER}))/$(basename {DRAFT_ADAPTER})
!ls -la /kaggle/working

total 163636
drwxr-xr-x  4 root root      4096 Nov  5 13:44 .
drwxr-xr-x  5 root root      4096 Nov  5 12:44 ..
-rw-r--r--  1 root root 166019437 Nov  5 13:44 m3_adapters.zip
-rw-r--r--  1 root root   1522453 Nov  5 13:44 m3_results.zip
drwxr-xr-x 12 root root      4096 Nov  5 12:50 trial-eligibility-transformer
drwxr-xr-x  2 root root      4096 Nov  5 12:44 .virtual_documents
